In [19]:
"""
financial_audit_system.py
==========================
Complete 4-agent financial audit pipeline.

Pipeline
--------
    CSV
     │
     ▼
  DataAgent          — load, validate, clean
     │  DataFrame
     ▼
  AnomalyAgent       — 8 detection rules, dynamic confidence
     │  list[Finding]
     ▼
  AuditTrailAgent    — aggregate, score, rank
     │  list[AuditRecord]
     ▼
  DecisionAgent      — recommended action + explanation

Usage
-----
    from financial_audit_system import run_pipeline
    decisions = run_pipeline("transactions.csv")
    for d in decisions:
        print(d["transaction_id"], d["risk_level"], d["recommended_action"])
"""

from __future__ import annotations

import logging
from collections import defaultdict
from pathlib import Path
from typing import Any

import pandas as pd

# ---------------------------------------------------------------------------
# Logging — single shared config
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


# ===========================================================================
# AGENT 1 — DataAgent
# ===========================================================================
class DataAgent:
    """
    Ingests, validates, and cleans the raw transaction CSV.

    Responsibilities
    ----------------
    - Load CSV with all columns as strings to prevent silent type coercion.
    - Assert every required column is present.
    - Convert date columns to datetime64.
    - Convert numeric columns to float64.
    - Fill missing / blank GSTIN values with "UNKNOWN".
    - Add boolean flag ``po_missing`` for transactions without a PO number.

    Parameters
    ----------
    filepath : str | Path
        Path to the transactions CSV file.
    """

    REQUIRED_COLUMNS: list[str] = [
        "transaction_id", "invoice_id", "transaction_date", "invoice_date",
        "due_date", "vendor_id", "vendor_name", "vendor_type", "category",
        "cost_center", "base_amount_inr", "gst_percent", "gst_amount_inr",
        "total_amount_inr", "currency", "payment_method", "payment_status",
        "created_by", "approver", "approval_level", "po_number",
        "contract_id", "invoice_present", "gstin",
    ]

    DATE_COLUMNS: list[str] = ["transaction_date", "invoice_date", "due_date"]

    NUMERIC_COLUMNS: list[str] = [
        "base_amount_inr", "gst_percent", "gst_amount_inr", "total_amount_inr",
    ]

    def __init__(self, filepath: str | Path) -> None:
        self.filepath = Path(filepath)
        self.logger = logging.getLogger(self.__class__.__name__)

    # ------------------------------------------------------------------
    # Public
    # ------------------------------------------------------------------

    def run(self) -> pd.DataFrame:
        """
        Execute the full data-preparation pipeline.

        Returns
        -------
        pd.DataFrame
            Clean, typed DataFrame ready for anomaly detection.

        Raises
        ------
        FileNotFoundError
            If *filepath* does not exist.
        ValueError
            If any required column is missing.
        """
        self.logger.info("Starting — file: %s", self.filepath)
        df = self._load()
        self._validate_columns(df)
        df = self._convert_dates(df)
        df = self._convert_numerics(df)
        df = self._handle_missing(df)
        self.logger.info("Done — %d rows, %d columns", df.shape[0], df.shape[1])
        return df

    # ------------------------------------------------------------------
    # Private
    # ------------------------------------------------------------------

    def _load(self) -> pd.DataFrame:
        """Read CSV; keep all values as strings initially."""
        if not self.filepath.exists():
            raise FileNotFoundError(f"File not found: {self.filepath}")
        df = pd.read_csv(self.filepath, dtype=str)
        df.columns = df.columns.str.strip()
        self.logger.info("Loaded %d rows from '%s'.", len(df), self.filepath.name)
        return df

    def _validate_columns(self, df: pd.DataFrame) -> None:
        """Raise ValueError listing every missing required column."""
        missing = [c for c in self.REQUIRED_COLUMNS if c not in df.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        self.logger.info("Column validation passed.")

    def _convert_dates(self, df: pd.DataFrame) -> pd.DataFrame:
        """Parse date columns; unparseable values become NaT."""
        for col in self.DATE_COLUMNS:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            nulls = df[col].isna().sum()
            if nulls:
                self.logger.warning("'%s': %d value(s) → NaT.", col, nulls)
        return df

    def _convert_numerics(self, df: pd.DataFrame) -> pd.DataFrame:
        """Coerce numeric columns to float64; non-numeric values become NaN."""
        for col in self.NUMERIC_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            nulls = df[col].isna().sum()
            if nulls:
                self.logger.warning("'%s': %d value(s) → NaN.", col, nulls)
        return df

    def _handle_missing(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply domain-specific missing-value rules for GSTIN and PO number."""
        # --- GSTIN: normalise and fill ---
        df["gstin"] = df["gstin"].astype(str).str.strip()
        gstin_blank = df["gstin"].isna() | df["gstin"].isin(["", "nan", "NaN", "None"])
        df.loc[gstin_blank, "gstin"] = "UNKNOWN"
        if gstin_blank.sum():
            self.logger.info("Filled %d blank GSTIN(s) with 'UNKNOWN'.", gstin_blank.sum())

        # --- PO Number: flag only, do NOT fill ---
        df["po_number"] = df["po_number"].astype(str).str.strip()
        df["po_missing"] = df["po_number"].isin(["", "nan", "NaN", "None"]) | df["po_number"].isna()
        if df["po_missing"].sum():
            self.logger.info("Flagged %d transaction(s) with missing PO.", df["po_missing"].sum())

        return df







In [20]:
# ===========================================================================
# AGENT 2 — AnomalyAgent
# ===========================================================================
class AnomalyAgent:
    """
    Detects financial anomalies using eight configurable rules.

    Rules
    -----
    1. DUPLICATE_TRANSACTION   — same invoice_id + vendor_id + amount
    2. AMOUNT_SPIKE            — > spike_multiplier × vendor average
    3. HIGH_VALUE_TRANSACTION  — above 90th-percentile threshold (dynamic)
    4. RAPID_TRANSACTIONS      — ≥ 4 transactions for same vendor in 1 day
    5. MISSING_INVOICE         — invoice_present == "No"
    6. MISSING_GSTIN           — gstin == "UNKNOWN"
    7. MISSING_PO              — po_missing flag is True
    8. APPROVAL_MISMATCH       — high-value transaction approved at L1

    Parameters
    ----------
    df : pd.DataFrame
        Clean DataFrame from DataAgent.
    spike_multiplier : float
        Amount-spike detection threshold (default 3×).
    high_value_percentile : float
        Percentile for dynamic high-value threshold (default 0.90).
    approval_threshold : float
        Minimum INR amount requiring L2/L3 approval (default 100 000).
    rapid_tx_window_days : int
        Rolling window in days for rapid-transaction detection (default 1).
    rapid_tx_min_count : int
        Minimum burst size to trigger rapid-transaction flag (default 4).
    """

    def __init__(
        self,
        df: pd.DataFrame,
        spike_multiplier: float = 3.0,
        high_value_percentile: float = 0.90,
        approval_threshold: float = 100_000.0,
        rapid_tx_window_days: int = 1,
        rapid_tx_min_count: int = 4,
    ) -> None:
        self.df = df.copy()
        self.spike_multiplier = spike_multiplier
        self.high_value_percentile = high_value_percentile
        self.approval_threshold = approval_threshold
        self.rapid_tx_window_days = rapid_tx_window_days
        self.rapid_tx_min_count = rapid_tx_min_count
        self.logger = logging.getLogger(self.__class__.__name__)

    # ------------------------------------------------------------------
    # Public
    # ------------------------------------------------------------------

    def run(self) -> list[dict[str, Any]]:
        """
        Run all detection rules and return a flat list of findings.

        Returns
        -------
        list[dict]
            Each dict contains: transaction_id, issue_type, severity,
            confidence (float 0–1), reason.
        """
        self.logger.info("Starting — %d transactions.", len(self.df))
        findings: list[dict[str, Any]] = []
        findings += self._detect_duplicates()
        findings += self._detect_amount_spike()
        findings += self._detect_high_value()
        findings += self._detect_rapid_transactions()
        findings += self._detect_missing_invoice()
        findings += self._detect_missing_gstin()
        findings += self._detect_missing_po()
        findings += self._detect_approval_mismatch()
        self.logger.info("Done — %d finding(s) across %d rule(s).", len(findings), 8)
        return findings

    # ------------------------------------------------------------------
    # Rule helpers
    # ------------------------------------------------------------------

    @staticmethod
    def _finding(
        tx_id: str,
        issue_type: str,
        severity: str,
        confidence: float,
        reason: str,
    ) -> dict[str, Any]:
        """Construct a standardised finding dictionary."""
        return {
            "transaction_id": tx_id,
            "issue_type": issue_type,
            "severity": severity,
            "confidence": round(confidence, 2),
            "reason": reason,
        }

    # ------------------------------------------------------------------
    # Rule 1 — Duplicate Transaction
    # ------------------------------------------------------------------

    def _detect_duplicates(self) -> list[dict[str, Any]]:
        """
        Flag transactions sharing the same invoice_id, vendor_id, and
        base_amount_inr — strong signal of double-payment or replay attack.
        """
        findings: list[dict[str, Any]] = []
        dup_mask = self.df.duplicated(
            subset=["invoice_id", "vendor_id", "base_amount_inr"],
            keep=False,
        )
        for _, row in self.df[dup_mask].iterrows():
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="DUPLICATE_TRANSACTION",
                severity="high",
                confidence=0.95,
                reason=(
                    f"Invoice '{row['invoice_id']}' appears more than once for vendor "
                    f"'{row['vendor_name']}' with an identical amount of "
                    f"₹{row['base_amount_inr']:,.2f}. Possible duplicate payment."
                ),
            ))
        self.logger.info("DUPLICATE_TRANSACTION — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 2 — Amount Spike
    # ------------------------------------------------------------------

    def _detect_amount_spike(self) -> list[dict[str, Any]]:
        """
        Flag transactions where base_amount_inr exceeds spike_multiplier ×
        the vendor's mean transaction amount.  Confidence scales with ratio.
        """
        findings: list[dict[str, Any]] = []
        vendor_avg = self.df.groupby("vendor_id")["base_amount_inr"].transform("mean")
        mask = self.df["base_amount_inr"] > self.spike_multiplier * vendor_avg

        for _, row in self.df[mask].iterrows():
            avg = vendor_avg.loc[row.name]
            ratio = row["base_amount_inr"] / avg if avg > 0 else 0
            confidence = 0.95 if ratio > 5 else (0.88 if ratio > 4 else 0.80)
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="AMOUNT_SPIKE",
                severity="high",
                confidence=confidence,
                reason=(
                    f"₹{row['base_amount_inr']:,.2f} is {ratio:.2f}× the vendor average "
                    f"of ₹{avg:,.2f} for '{row['vendor_name']}' "
                    f"(threshold: {self.spike_multiplier}×)."
                ),
            ))
        self.logger.info("AMOUNT_SPIKE — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 3 — High Value Transaction (dynamic 90th-percentile)
    # ------------------------------------------------------------------

    def _detect_high_value(self) -> list[dict[str, Any]]:
        """
        Flag transactions above the dataset's 90th-percentile amount.
        Confidence scales proportionally with how far the amount exceeds
        the threshold (capped at 0.95).
        """
        findings: list[dict[str, Any]] = []
        threshold = self.df["base_amount_inr"].quantile(self.high_value_percentile)
        mask = self.df["base_amount_inr"] > threshold

        for _, row in self.df[mask].iterrows():
            amt = row["base_amount_inr"]
            confidence = min(0.95, round(amt / threshold, 2))
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="HIGH_VALUE_TRANSACTION",
                severity="medium",
                confidence=confidence,
                reason=(
                    f"₹{amt:,.2f} exceeds the dynamic {int(self.high_value_percentile * 100)}th-"
                    f"percentile threshold of ₹{threshold:,.2f}. "
                    f"Amount is {amt / threshold:.2f}× the threshold."
                ),
            ))
        self.logger.info("HIGH_VALUE_TRANSACTION — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 4 — Rapid Transactions
    # ------------------------------------------------------------------

    def _detect_rapid_transactions(self) -> list[dict[str, Any]]:
        """
        Flag vendors with ≥ rapid_tx_min_count transactions within
        rapid_tx_window_days.  Each unique transaction is flagged once.
        Confidence increases with burst size.
        """
        findings: list[dict[str, Any]] = []
        seen: set[str] = set()

        valid = self.df.dropna(subset=["transaction_date"]).copy()
        valid = valid.sort_values(["vendor_id", "transaction_date"])

        for vendor_id, group in valid.groupby("vendor_id"):
            group = group.reset_index(drop=True)
            n = len(group)
            dates = group["transaction_date"]
            tx_ids = group["transaction_id"]
            vendor_name = group["vendor_name"].iloc[0]

            for i in range(n):
                burst = [i]
                for j in range(i + 1, n):
                    delta = (dates.iloc[j] - dates.iloc[i]).days
                    if delta <= self.rapid_tx_window_days:
                        burst.append(j)
                    else:
                        break

                if len(burst) >= self.rapid_tx_min_count:
                    burst_size = len(burst)
                    confidence = 0.90 if burst_size >= 6 else 0.82
                    burst_dates = ", ".join(
                        str(dates.iloc[k].date()) for k in burst
                    )
                    for idx in burst:
                        tx = tx_ids.iloc[idx]
                        if tx not in seen:
                            seen.add(tx)
                            findings.append(self._finding(
                                tx_id=tx,
                                issue_type="RAPID_TRANSACTIONS",
                                severity="medium",
                                confidence=confidence,
                                reason=(
                                    f"Vendor '{vendor_name}' has {burst_size} transaction(s) "
                                    f"within {self.rapid_tx_window_days} day(s) "
                                    f"(dates: {burst_dates}). "
                                    f"Possible split-payment or fraud pattern."
                                ),
                            ))

        self.logger.info("RAPID_TRANSACTIONS — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 5 — Missing Invoice
    # ------------------------------------------------------------------

    def _detect_missing_invoice(self) -> list[dict[str, Any]]:
        """Flag transactions processed without a supporting invoice document."""
        findings: list[dict[str, Any]] = []
        mask = self.df["invoice_present"].str.strip().str.lower() == "no"

        for _, row in self.df[mask].iterrows():
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="MISSING_INVOICE",
                severity="high",
                confidence=0.95,
                reason=(
                    f"Transaction ₹{row['base_amount_inr']:,.2f} to '{row['vendor_name']}' "
                    f"was processed without an attached invoice. "
                    f"Violates procurement policy."
                ),
            ))
        self.logger.info("MISSING_INVOICE — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 6 — Missing GSTIN
    # ------------------------------------------------------------------

    def _detect_missing_gstin(self) -> list[dict[str, Any]]:
        """Flag transactions where vendor GSTIN is absent (filled as UNKNOWN by DataAgent)."""
        findings: list[dict[str, Any]] = []
        mask = self.df["gstin"] == "UNKNOWN"

        for _, row in self.df[mask].iterrows():
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="MISSING_GSTIN",
                severity="medium",
                confidence=0.85,
                reason=(
                    f"Vendor '{row['vendor_name']}' has no registered GSTIN on record. "
                    f"GST input credit of ₹{row['gst_amount_inr']:,.2f} may not be claimable."
                ),
            ))
        self.logger.info("MISSING_GSTIN — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 7 — Missing PO
    # ------------------------------------------------------------------

    def _detect_missing_po(self) -> list[dict[str, Any]]:
        """Flag transactions that have no associated Purchase Order number."""
        findings: list[dict[str, Any]] = []
        mask = self.df["po_missing"]

        for _, row in self.df[mask].iterrows():
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="MISSING_PO",
                severity="medium",
                confidence=0.85,
                reason=(
                    f"No Purchase Order number linked to transaction ₹{row['base_amount_inr']:,.2f} "
                    f"for vendor '{row['vendor_name']}'. "
                    f"May indicate unauthorised spend."
                ),
            ))
        self.logger.info("MISSING_PO — %d finding(s).", len(findings))
        return findings

    # ------------------------------------------------------------------
    # Rule 8 — Approval Mismatch
    # ------------------------------------------------------------------

    def _detect_approval_mismatch(self) -> list[dict[str, Any]]:
        """
        Flag high-value transactions (above approval_threshold) that were
        approved at L1 — the lowest approval level.

        Confidence scales with how much the amount exceeds the threshold.
        """
        findings: list[dict[str, Any]] = []

        mask = (
            (self.df["base_amount_inr"] > self.approval_threshold) &
            (self.df["approval_level"].str.strip().str.upper() == "L1")
        )

        for _, row in self.df[mask].iterrows():
            ratio = row["base_amount_inr"] / self.approval_threshold
            confidence = min(0.97, round(0.88 + (ratio - 1) * 0.04, 2))
            findings.append(self._finding(
                tx_id=row["transaction_id"],
                issue_type="APPROVAL_MISMATCH",
                severity="high",
                confidence=confidence,
                reason=(
                    f"Transaction of ₹{row['base_amount_inr']:,.2f} "
                    f"({ratio:.2f}× the L2 threshold of ₹{self.approval_threshold:,.0f}) "
                    f"was approved by '{row['approver']}' at L1. "
                    f"Requires L2 or higher authorisation."
                ),
            ))
        self.logger.info("APPROVAL_MISMATCH — %d finding(s).", len(findings))
        return findings


In [21]:
# ===========================================================================
# AGENT 3 — AuditTrailAgent
# ===========================================================================

# Severity weights for risk scoring
_SEVERITY_WEIGHTS: dict[str, int] = {"high": 3, "medium": 2, "low": 1}

# Risk level thresholds (inclusive lower bound, checked in order)
_RISK_THRESHOLDS: list[tuple[int, str]] = [
    (5, "HIGH RISK"),
    (3, "MEDIUM RISK"),
    (0, "LOW RISK"),
]


class AuditTrailAgent:
    """
    Aggregates per-finding anomaly results into per-transaction audit records.

    For each transaction the agent computes:
    - ``issues``          — full detail list (type, severity, confidence, reason)
    - ``risk_score``      — sum of severity weights (high=3, medium=2, low=1)
    - ``risk_level``      — HIGH / MEDIUM / LOW RISK based on score thresholds
    - ``top_issue``       — issue_type with highest severity (then confidence)
    - ``avg_confidence``  — mean confidence across all issues
    - ``summary``         — plain-English one-liner

    Parameters
    ----------
    findings : list[dict]
        Raw findings from AnomalyAgent.  Each must contain:
        transaction_id, issue_type, severity, confidence, reason.
    """

    def __init__(self, findings: list[dict[str, Any]]) -> None:
        self.findings = findings
        self.logger = logging.getLogger(self.__class__.__name__)

    # ------------------------------------------------------------------
    # Public
    # ------------------------------------------------------------------

    def run(self) -> list[dict[str, Any]]:
        """
        Build and return per-transaction audit records sorted by descending
        risk_score (most critical first).

        Returns
        -------
        list[dict]
            Audit records with keys: transaction_id, issues, risk_score,
            risk_level, top_issue, avg_confidence, summary.
        """
        self.logger.info("Starting — %d raw finding(s).", len(self.findings))
        grouped = self._group()
        records = [self._build_record(tx_id, issues) for tx_id, issues in grouped.items()]
        records.sort(key=lambda r: r["risk_score"], reverse=True)
        self.logger.info("Done — %d unique transaction(s) audited.", len(records))
        return records

    # ------------------------------------------------------------------
    # Private
    # ------------------------------------------------------------------

    def _group(self) -> dict[str, list[dict[str, Any]]]:
        """Partition raw findings by transaction_id."""
        grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
        for f in self.findings:
            grouped[f["transaction_id"]].append(f)
        return dict(grouped)

    def _build_record(
        self, tx_id: str, raw_findings: list[dict[str, Any]]
    ) -> dict[str, Any]:
        """Construct a single audit record from grouped findings."""
        issues = [
            {
                "issue_type": f["issue_type"],
                "severity":   f["severity"],
                "confidence": f["confidence"],
                "reason":     f["reason"],
            }
            for f in raw_findings
        ]

        risk_score  = sum(_SEVERITY_WEIGHTS.get(i["severity"], 1) for i in issues)
        risk_level  = self._assign_level(risk_score)
        top_issue   = self._top_issue(issues)
        avg_conf    = round(sum(i["confidence"] for i in issues) / len(issues), 3)
        summary     = f"{len(issues)} issue(s): {', '.join(i['issue_type'] for i in issues)}"

        return {
            "transaction_id": tx_id,
            "issues":         issues,
            "risk_score":     risk_score,
            "risk_level":     risk_level,
            "top_issue":      top_issue,
            "avg_confidence": avg_conf,
            "summary":        summary,
        }

    @staticmethod
    def _assign_level(score: int) -> str:
        """Map a numeric risk score to a categorical risk level."""
        for threshold, label in _RISK_THRESHOLDS:
            if score >= threshold:
                return label
        return "LOW RISK"

    @staticmethod
    def _top_issue(issues: list[dict[str, Any]]) -> str:
        """
        Return the issue_type that represents the greatest risk.
        Primary sort: severity weight (desc); secondary: confidence (desc).
        """
        ranked = sorted(
            issues,
            key=lambda i: (
                _SEVERITY_WEIGHTS.get(i["severity"], 1),
                i["confidence"],
            ),
            reverse=True,
        )
        return ranked[0]["issue_type"]



In [25]:
from typing import Any
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(name)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# ---------------------------------------------------------------------------
# Issue-type → plain-English label used in explanations
# ---------------------------------------------------------------------------
_ISSUE_CONTEXT: dict[str, str] = {
    "DUPLICATE_TRANSACTION":  "a suspected duplicate payment",
    "AMOUNT_SPIKE":           "an abnormal amount spike vs vendor history",
    "HIGH_VALUE_TRANSACTION": "a high-value transaction above normal thresholds",
    "RAPID_TRANSACTIONS":     "a burst of rapid transactions from the same vendor",
    "MISSING_INVOICE":        "a missing invoice document",
    "MISSING_GSTIN":          "an unverified vendor GSTIN",
    "MISSING_PO":             "a missing Purchase Order reference",
    "APPROVAL_MISMATCH":      "an approval level mismatch for a high-value transaction",
}

# Risk level → recommended action
_ACTIONS: dict[str, str] = {
    "HIGH RISK":   "Block transaction and initiate audit review",
    "MEDIUM RISK": "Flag for manual review",
    "LOW RISK":    "No immediate action required",
}


class DecisionAgent:
    """
    Translates AuditTrailAgent records into recommended actions
    with human-readable explanations.

    Input  : list of audit records from AuditTrailAgent.run()
    Output : list of decision dicts, sorted by descending risk_score

    Each output dict contains:
        transaction_id     — str
        risk_level         — "HIGH RISK" / "MEDIUM RISK" / "LOW RISK"
        risk_score         — int
        recommended_action — str
        explanation        — str  (full narrative)
        top_issue          — str  (highest-priority issue type)
        avg_confidence     — float
    """

    def __init__(self, audit_records: list[dict[str, Any]]) -> None:
        self.audit_records = audit_records
        self.logger = logging.getLogger(self.__class__.__name__)

    # ------------------------------------------------------------------
    # Public
    # ------------------------------------------------------------------

    def run(self) -> list[dict[str, Any]]:
        """Run all records through the decision logic and return sorted output."""
        self.logger.info("DecisionAgent starting — %d audit record(s).", len(self.audit_records))

        decisions = [self._decide(record) for record in self.audit_records]
        decisions.sort(key=lambda d: d["risk_score"], reverse=True)

        self.logger.info("DecisionAgent done — %d decision(s) generated.", len(decisions))
        return decisions

    # ------------------------------------------------------------------
    # Private
    # ------------------------------------------------------------------

    def _decide(self, record: dict[str, Any]) -> dict[str, Any]:
        """Build one decision dict from one audit record."""
        risk_level  = record["risk_level"]
        issues      = record["issues"]
        action      = _ACTIONS.get(risk_level, "No immediate action required")
        explanation = self._build_explanation(record, risk_level, issues)

        return {
            "transaction_id":     record["transaction_id"],
            "risk_level":         risk_level,
            "risk_score":         record["risk_score"],
            "recommended_action": action,
            "explanation":        explanation,
            "top_issue":          record.get("top_issue", "N/A"),
            "avg_confidence":     record.get("avg_confidence", 0.0),
        }

    @staticmethod
    def _build_explanation(
        record: dict[str, Any],
        risk_level: str,
        issues: list[dict[str, Any]],
    ) -> str:
        """Compose a full narrative explanation from risk level + issue details."""
        tx_id       = record["transaction_id"]
        issue_count = len(issues)
        top         = record.get("top_issue", issues[0]["issue_type"])

        # Pull the reason text for the top issue
        top_reason = next(
            (i["reason"] for i in issues if i["issue_type"] == top),
            "See issues list for details.",
        )

        # Human-readable labels for each issue
        issue_labels = [
            _ISSUE_CONTEXT.get(i["issue_type"], i["issue_type"].lower().replace("_", " "))
            for i in issues
        ]

        # Action-specific closing sentence
        if risk_level == "HIGH RISK":
            action_detail = (
                "This transaction has been blocked pending a full audit review. "
                "Finance and compliance teams must be notified immediately."
            )
        elif risk_level == "MEDIUM RISK":
            action_detail = (
                "This transaction has been routed to the manual review queue. "
                "A senior approver should verify before processing."
            )
        else:
            action_detail = (
                "This transaction shows minor irregularities that are logged for "
                "record-keeping but require no immediate intervention."
            )

        return (
            f"Transaction {tx_id} triggered {issue_count} issue(s): "
            f"{'; '.join(issue_labels)}. "
            f"Primary concern — {top_reason} "
            f"{action_detail}"
        )




In [27]:
# ---------------------------------------------------------------------------
# Run the full pipeline
# ---------------------------------------------------------------------------
df        = DataAgent("transactions.csv").run()
findings  = AnomalyAgent(df).run()
audit     = AuditTrailAgent(findings).run()
decisions = DecisionAgent(audit).run()

# ---------------------------------------------------------------------------
# 1. DECISIONS TABLE
# ---------------------------------------------------------------------------
import pandas as pd

decisions_df = pd.DataFrame(decisions)[[
    "transaction_id", "risk_level", "risk_score",
    "top_issue", "avg_confidence", "recommended_action"
]]

print("=" * 70)
print("  DECISION AGENT — OUTPUT")
print("=" * 70)
print(f"  Total transactions flagged : {len(decisions_df)}")
print(f"  HIGH RISK                  : {(decisions_df['risk_level'] == 'HIGH RISK').sum()}")
print(f"  MEDIUM RISK                : {(decisions_df['risk_level'] == 'MEDIUM RISK').sum()}")
print(f"  LOW RISK                   : {(decisions_df['risk_level'] == 'LOW RISK').sum()}")
print("=" * 70)

decisions_df

# ---------------------------------------------------------------------------
# 2. FULL AUDIT TRAIL TABLE
# ---------------------------------------------------------------------------
audit_df = pd.DataFrame(audit)[[
    "transaction_id", "risk_score", "risk_level",
    "top_issue", "avg_confidence", "summary"
]]

print("\n" + "=" * 70)
print("  AUDIT TRAIL — SUMMARY TABLE")
print("=" * 70)
audit_df

# ---------------------------------------------------------------------------
# 3. HIGH RISK — DETAILED BREAKDOWN
# ---------------------------------------------------------------------------
high_risk = [d for d in decisions if d["risk_level"] == "HIGH RISK"]

print("\n" + "=" * 70)
print("  HIGH RISK TRANSACTIONS — FULL DETAIL")
print("=" * 70)

for d in high_risk:
    print(f"\n  Transaction  : {d['transaction_id']}")
    print(f"  Risk Level   : {d['risk_level']}  (score: {d['risk_score']})")
    print(f"  Top Issue    : {d['top_issue']}")
    print(f"  Avg Conf     : {d['avg_confidence']}")
    print(f"  Action       : {d['recommended_action']}")
    print(f"  Explanation  : {d['explanation']}")
    print("-" * 70)

# ---------------------------------------------------------------------------
# 4. ANOMALY FINDINGS TABLE
# ---------------------------------------------------------------------------
findings_df = pd.DataFrame(findings)

print("\n" + "=" * 70)
print("  ANOMALY FINDINGS — BY ISSUE TYPE")
print("=" * 70)
print(findings_df["issue_type"].value_counts().to_string())
print()

findings_df

# ---------------------------------------------------------------------------
# 5. RISK LEVEL DISTRIBUTION
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("  RISK LEVEL DISTRIBUTION")
print("=" * 70)
print(decisions_df["risk_level"].value_counts().to_string())
print()
print(decisions_df["top_issue"].value_counts().to_string())

  DECISION AGENT — OUTPUT
  Total transactions flagged : 56
  HIGH RISK                  : 6
  MEDIUM RISK                : 17
  LOW RISK                   : 33

  AUDIT TRAIL — SUMMARY TABLE

  HIGH RISK TRANSACTIONS — FULL DETAIL

  Transaction  : TXN2026012
  Risk Level   : HIGH RISK  (score: 8)
  Top Issue    : MISSING_INVOICE
  Avg Conf     : 0.9
  Action       : Block transaction and initiate audit review
  Explanation  : Transaction TXN2026012 triggered 3 issue(s): an abnormal amount spike vs vendor history; a high-value transaction above normal thresholds; a missing invoice document. Primary concern — Transaction ₹247,437.86 to 'BuildRight Contractors' was processed without an attached invoice. Violates procurement policy. This transaction has been blocked pending a full audit review. Finance and compliance teams must be notified immediately.
----------------------------------------------------------------------

  Transaction  : TXN2026155
  Risk Level   : HIGH RISK  (score: 7

In [28]:
final_df = pd.DataFrame(decisions)
final_df.head(10)

,transaction_id,risk_level,risk_score,recommended_action,explanation,top_issue,avg_confidence
0,TXN2026012,HIGH RISK,8,Block transaction and initiate audit review,Transaction TXN2026012 triggered 3 issue(s): a...,MISSING_INVOICE,0.900
1,TXN2026155,HIGH RISK,7,Block transaction and initiate audit review,Transaction TXN2026155 triggered 3 issue(s): a...,MISSING_INVOICE,0.917
2,TXN2026059,HIGH RISK,5,Block transaction and initiate audit review,Transaction TXN2026059 triggered 2 issue(s): a...,AMOUNT_SPIKE,0.875
3,TXN2026075,HIGH RISK,5,Block transaction and initiate audit review,Transaction TXN2026075 triggered 2 issue(s): a...,AMOUNT_SPIKE,0.915
4,TXN2026148,HIGH RISK,5,Block transaction and initiate audit review,Transaction TXN2026148 triggered 2 issue(s): a...,AMOUNT_SPIKE,0.915
5,TXN2026198,HIGH RISK,5,Block transaction and initiate audit review,Transaction TXN2026198 triggered 2 issue(s): a...,AMOUNT_SPIKE,0.915
6,TXN2026027,MEDIUM RISK,4,Flag for manual review,Transaction TXN2026027 triggered 2 issue(s): a...,HIGH_VALUE_TRANSACTION,0.900
7,TXN2026057,MEDIUM RISK,4,Flag for manual review,Transaction TXN2026057 triggered 2 issue(s): a...,HIGH_VALUE_TRANSACTION,0.900
8,TXN2026151,MEDIUM RISK,4,Flag for manual review,Transaction TXN2026151 triggered 2 issue(s): a...,HIGH_VALUE_TRANSACTION,0.900
9,TXN2026196,MEDIUM RISK,4,Flag for manual review,Transaction TXN2026196 triggered 2 issue(s): a...,HIGH_VALUE_TRANSACTION,0.900
